In [2]:
import dropbox
import io
import json
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np

# cosmos
jwst_url = 'https://www.dropbox.com/scl/fo/adjat4nzyut3zijz8b6i3/ACOIMQ2yHcRoTdkfayMh0XE?rlkey=qamhxolhsmz05hs747aiiencm&dl=0'
nisp_url = 'https://www.dropbox.com/scl/fo/kej9smd6kjfqd3e1xj3pn/AN-77wdF13ups2AFvNuCpsw?rlkey=sludiaw7bb7fn82q20p9glwr2&dl=0'

# Initialize Dropbox
home = "/Users/shemmati"
with open(f"{home}/secrets/dropbox_token") as token_file:
    token = token_file.read().strip()

dbx = dropbox.Dropbox(token)


def extract_fits_footprint(dbx, shared_url, file_name, buffer=0.0001):
    """Extracts the RA/DEC footprint of a FITS file and adds a small buffer to avoid edge issues."""
    try:
        _, res = dbx.sharing_get_shared_link_file(url=shared_url, path=f"/{file_name}")
        fits_data = io.BytesIO(res.content)

        with fits.open(fits_data, memmap=True) as hdul:
            header = hdul[0].header
            wcs = WCS(header)

            naxis1, naxis2 = header["NAXIS1"], header["NAXIS2"]

            # Get RA/DEC for the four corners of the image
            ra_dec_corners = wcs.all_pix2world(
                [[0, 0], [0, naxis2], [naxis1, 0], [naxis1, naxis2]], 1
            )

            ra_vals, dec_vals = zip(*ra_dec_corners)

            return {
                "file_name": file_name,
                "ra_min": min(ra_vals) + buffer,
                "ra_max": max(ra_vals) - buffer,
                "dec_min": min(dec_vals) + buffer,
                "dec_max": max(dec_vals) - buffer,
            }

    except Exception as e:
        print(f"Skipping {file_name} due to error: {e}")
        return None

def build_fits_index(dbx, shared_url, output_file):
    """Creates a JSON file with RA/DEC footprints of all FITS files."""
    shared_link = dropbox.files.SharedLink(url=shared_url)
    try:
        folder_metadata = dbx.files_list_folder(path="", shared_link=shared_link)
        fits_files = [file.name for file in folder_metadata.entries if file.name.endswith(".fits")]
    except dropbox.exceptions.ApiError as e:
        print(f"Error accessing shared folder: {e}")
        return

    index = []
    for file_name in fits_files:
        print(f"Processing {file_name}...")
        footprint = extract_fits_footprint(dbx, shared_url, file_name)
        if footprint:
            index.append(footprint)

    # Save index as JSON
    with open(output_file, "w") as f:
        json.dump(index, f)

    print(f"Saved index to {output_file}")


In [4]:
# Run once to generate footprint index
build_fits_index(dbx, nisp_url, "nisp_index.json")


Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101547136-35B51_20240806T001819.360855Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101542815-6B457E_20240805T203456.732184Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101542818-8735E3_20240806T005419.796340Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101545698-2BE092_20240806T041336.046711Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE102086074-E195A1_20250305T113632.411211Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101545695-10BEF4_20240806T094436.373900Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101548574-989304_20240806T083200.710577Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101542816-4B5425_20240805T234610.183372Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101547137-19890F_20240806T134740.698407Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE101545699-2C114E_20240806T024938.493631Z_00.00.fits...
Processing EUC_MER_BGSUB-MOSAIC

In [22]:
build_fits_index(dbx, jwst_url, "jwst_index.json")


Processing mosaic_nircam_f115w_COSMOS-Web_60mas_A4_v0_8_sci.fits...


Set DATE-AVG to '2023-04-29T00:59:18.908' from MJD-AVG.
Set DATE-END to '2024-04-17T18:10:49.417' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     1.572297 from OBSGEO-[XYZ].
Set OBSGEO-H to 1265954850.776 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Processing mosaic_nircam_f115w_COSMOS-Web_60mas_A2_v0_8_sci.fits...


Set DATE-AVG to '2023-05-08T21:45:22.508' from MJD-AVG.
Set DATE-END to '2024-04-23T13:30:58.404' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -14.819600 from OBSGEO-[XYZ].
Set OBSGEO-H to 1446438097.324 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Processing mosaic_nircam_f115w_COSMOS-Web_60mas_A3_v0_8_sci.fits...


...

Processing mosaic_nircam_f115w_COSMOS-Web_60mas_B2_v0_8_sci.fits...


Set DATE-AVG to '2023-12-05T00:48:43.596' from MJD-AVG.
Set DATE-END to '2024-01-06T14:01:01.738' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.876254 from OBSGEO-[XYZ].
Set OBSGEO-H to 1689778862.210 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Processing mosaic_nircam_f115w_COSMOS-Web_60mas_B3_v0_8_sci.fits...


Set DATE-AVG to '2024-01-11T09:18:21.208' from MJD-AVG.
Set DATE-END to '2024-04-17T15:32:40.723' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.991008 from OBSGEO-[XYZ].
Set OBSGEO-H to 1696578874.448 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Processing mosaic_nircam_f115w_COSMOS-Web_60mas_B6_v0_8_sci.fits...
Skipping mosaic_nircam_f115w_COSMOS-Web_60mas_B6_v0_8_sci.fits due to error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Saved index to jwst_index.json
